# Practice Lab 4.4 - Vector Stores (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 4 - Lesson 4.4 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-4.4.html` (8 exercises).
Each exercise below repeats the prompt and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- **No API key needed to run.** The keyless guard in the next cells detects a missing
  `OPENAI_API_KEY` and swaps in a deterministic offline pseudo-embedding, so every cell runs
  end to end with no network and no cost. Set `OPENAI_API_KEY` to reproduce the real
  `text-embedding-3-small` numbers quoted in the lesson (~$5 for the 200-doc corpus).
- Several exercises ship as SQL against real Postgres + pgvector, or a Dockerised Qdrant.
  Colab cannot run those services in a sandbox cell, so the worked solution is the same
  retrieval contract in-process (NumPy + sklearn + Qdrant in-memory) - swap the
  implementation, keep the contract. The production SQL for each exercise is on the HTML page.
- Production context: AcmeCorp financial-services RAG over OCC bulletins + GDPR articles +
  insurance-regulator guidance; CoreWeave / GCP `us-east1` for GDPR residency; USD costs.

In [ ]:
!pip install --quiet openai qdrant-client numpy scikit-learn


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    pass

# Keyless-runnable guard: this notebook runs end-to-end with NO OPENAI_API_KEY set.
# With a key it uses real OpenAI embeddings; without one it uses a deterministic
# offline pseudo-embedding (defined in the corpus/embed cell below).
HAS_OPENAI = bool(os.environ.get('OPENAI_API_KEY'))

print('OpenAI set:', HAS_OPENAI)
print('Mode:', 'LIVE OpenAI embeddings' if HAS_OPENAI else 'OFFLINE deterministic pseudo-embeddings (keyless)')
print('Note: this notebook does not require Pinecone or a real Postgres - both are simulated in-process so the cells run anywhere.')


---
## Shared setup - the AcmeCorp corpus every exercise queries

Every exercise below reads from one 200-doc synthetic corpus (US OCC bulletins + EU GDPR
articles + insurance-regulator cyber guidance) and one `embed()` helper. Run this once; the
eight exercise cells build on it. With no key, `embed()` returns a deterministic offline
pseudo-embedding so the whole notebook runs anywhere.

In [ ]:
# 200-doc synthetic AcmeCorp financial-services corpus - US OCC + EU GDPR + insurance-regulator guidance
import hashlib
import random
from types import SimpleNamespace

import numpy as np

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

random.seed(42)

occ_docs = [
    'OCC Bulletin 2011-12 on model risk management - AcmeCorp must maintain effective challenge and independent validation for any credit-scoring model.',
    'OCC Comptroller Handbook on Retail Lending - unsecured exposure limits and underwriting standards for consumer loan portfolios.',
    'OCC Bulletin 2013-29 on third-party relationships - cloud and AI vendors used by AcmeCorp require ongoing due-diligence and exit planning.',
    'OCC guidance on AI in credit decisions (2025) recommends human-in-the-loop review for any adverse action over $25,000.',
    'OCC Bulletin 2021-40 on operational resilience - AcmeCorp must document recovery objectives for its core lending platform.',
    'OCC Comptroller Handbook on Customer Complaints - grievance-redress turnaround should not exceed 30 days from receipt.',
    'OCC guidance on digital lending - all loan disbursements must flow through the regulated banking entity, no unlicensed pass-through.',
    'OCC Bulletin 2020-10 on cloud computing - the provider must support a compliant region such as GCP us-east1 or europe-west1.',
    'OCC model-documentation guidance section 4.2 - model records must include training-data lineage and an evaluation harness.',
    'OCC guidance on suspicious-activity reporting - fraud losses above $1,000,000 must be reported within 7 days.'
] * 4

gdpr_docs = [
    'GDPR Article 6 - processing is lawful only with a valid basis such as consent, contract, legal obligation, or legitimate interest.',
    'GDPR Article 7 - consent of the data subject must be freely given, specific, informed, and unambiguous, and as easy to withdraw as to give.',
    'GDPR Article 44 - the general principle for cross-border transfer: personal data may leave the EEA only with adequate safeguards.',
    'GDPR Article 15 - the data subject has the right of access to a copy of the personal data being processed.',
    'GDPR Articles 16 and 17 - the data subject has the right to rectification and the right to erasure of personal data.',
    'GDPR Article 6(1)(f) - legitimate interest may justify processing necessary to establish, exercise, or defend a legal claim.',
    'GDPR Article 37 - large-scale processing of financial data requires appointment of a Data Protection Officer.',
    'GDPR Article 5 - the storage-limitation principle: personal data must not be kept longer than necessary for the stated purpose.',
    'GDPR Article 77 - a data subject may lodge a complaint with a supervisory authority such as a national data-protection board.',
    'GDPR cross-border transfer guidance - AcmeCorp is advised to keep personal data resident on CoreWeave or GCP us-east1.'
] * 4

insreg_docs = [
    'Insurance-regulator cyber-security guidelines (2023) - insurance affiliates of AcmeCorp must maintain ISO 27001 certification.',
    'Insurance-regulator model-governance circular (2024) - AI pricing and claims models require documented validation and bias testing.',
    'Insurance-regulator product-filing pathway - standard health and motor products may use an expedited use-and-file approval route.',
    'Insurance-regulator conduct rules - claims turnaround and fair-treatment metrics must be reported to the supervisor quarterly.'
] * 5

filler_docs = [
    f'AcmeCorp internal SOP document {i}: branch-operations topic {i}, unrelated to the OCC bulletins or GDPR articles.'
    for i in range(200 - len(occ_docs) - len(gdpr_docs) - len(insreg_docs))
]

# Keep a source tag alongside each doc so the metadata-filtered-search cell has something to filter on.
corpus_pairs = (
    [(d, 'occ') for d in occ_docs]
    + [(d, 'gdpr') for d in gdpr_docs]
    + [(d, 'insurance') for d in insreg_docs]
    + [(d, 'filler') for d in filler_docs]
)
random.shuffle(corpus_pairs)
corpus = [p[0] for p in corpus_pairs]
corpus_sources = [p[1] for p in corpus_pairs]
print(f'Corpus size: {len(corpus)} docs')
print('Source mix:', {s: corpus_sources.count(s) for s in ('occ', 'gdpr', 'insurance', 'filler')})

# The OpenAI client is created ONLY when a key is present (keyless-runnable).
client = OpenAI() if (HAS_OPENAI and OpenAI is not None) else None

EMBED_DIM = 1536  # text-embedding-3-small dimensionality

def _keyless_embedding(text, dim=EMBED_DIM):
    """Deterministic offline pseudo-embedding: sha256 bag-of-words -> L2-normalized.
    Runs with NO OPENAI_API_KEY set. Shared tokens produce overlapping vectors, so the
    cosine-retrieval demo still surfaces topically-related docs."""
    vec = np.zeros(dim, dtype=np.float32)
    for token in str(text).lower().split():
        h = hashlib.sha256(token.encode('utf-8')).digest()  # 32 bytes
        for j in range(0, 32, 4):
            idx = int.from_bytes(h[j:j + 4], 'big') % dim
            sign = 1.0 if (h[j] & 1) == 0 else -1.0
            vec[idx] += sign
    norm = float(np.linalg.norm(vec))
    if norm > 0.0:
        vec /= norm
    return vec

def embeddings_create(inputs, model='text-embedding-3-small'):
    """Shape-compatible wrapper around client.embeddings.create that preserves the
    response shape (r.data[i].embedding). Uses OpenAI when HAS_OPENAI is set, else the
    deterministic offline fallback above."""
    batch = [inputs] if isinstance(inputs, str) else list(inputs)
    if client is not None:
        return client.embeddings.create(input=inputs, model=model)
    data = [SimpleNamespace(embedding=_keyless_embedding(t).tolist(), index=i)
            for i, t in enumerate(batch)]
    return SimpleNamespace(data=data, model=model + ':keyless-fallback')

def embed(text):
    r = embeddings_create(text, model='text-embedding-3-small')
    return np.array(r.data[0].embedding, dtype=np.float32)

if HAS_OPENAI:
    print('Embedding 200 docs with text-embedding-3-small (cost ~$5)...')
else:
    print('Embedding 200 docs with the deterministic offline fallback (no key, no cost)...')
corpus_vecs = np.vstack([embed(d) for d in corpus])
print(f'Embedding matrix shape: {corpus_vecs.shape}')

---
## Exercise 1 (Easy, 5 pts) - pgvector hello-world: install, insert, query

**Task.** Spin up Postgres 16 with the pgvector extension. Create a `circulars` table with a
1536-dim `embedding` column. Insert 100 synthetic OCC-bulletin chunks with
`text-embedding-3-small` embeddings. Run a cosine-similarity top-5 query.

**Success criteria** - rows inserted; the top-5 query returns in well under 50ms on a laptop;
the top result is the chunk most similar to the query.

The production SQL (`CREATE EXTENSION vector;` ... `ORDER BY embedding <=> $1 LIMIT 5`) is on
the HTML page. The keyless worked solution below is the same cosine contract in-process
(`PgvectorSim` over sklearn), so it runs with no Postgres and no key.

In [ ]:
# NumPy-backed pgvector simulator
from sklearn.neighbors import NearestNeighbors

class PgvectorSim:
    """Mimics pgvector cosine search via sklearn NearestNeighbors."""

    def __init__(self, dim: int):
        self.dim = dim
        self.rows = []  # list of (id, text, vec)
        self._index = None

    def upsert(self, doc_id: int, text: str, vec: np.ndarray) -> None:
        self.rows.append((doc_id, text, vec.astype(np.float32)))
        self._index = None  # invalidate

    def _build(self) -> None:
        if not self.rows:
            raise ValueError('no rows')
        mat = np.vstack([r[2] for r in self.rows])
        self._index = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='auto')
        self._index.fit(mat)
        self._mat = mat

    def query_cosine(self, qvec: np.ndarray, top_k: int = 5):
        if self._index is None:
            self._build()
        dists, idxs = self._index.kneighbors(qvec.reshape(1, -1), n_neighbors=top_k)
        return [(self.rows[i][0], self.rows[i][1], 1.0 - float(d)) for d, i in zip(dists[0], idxs[0])]

pg = PgvectorSim(dim=corpus_vecs.shape[1])
for i, (txt, vec) in enumerate(zip(corpus, corpus_vecs)):
    pg.upsert(i, txt, vec)

query = 'GDPR cross-border data transfer AcmeCorp'
qvec = embed(query)
print(f'pgvector-sim top-5 for: "{query}"')
for doc_id, text, sim in pg.query_cosine(qvec, top_k=5):
    print(f'  id={doc_id:3d} sim={sim:.3f} :: {text[:90]}...')

---
## Exercise 2 (Easy, 5 pts) - HNSW vs IVF: build time and query latency

**Task.** Take a synthetic embedding table. Build two indexes - one HNSW (m=16,
ef_construction=64), one IVF (lists=100). Measure build time and query latency for each.

**Success criteria** - both timings reported honestly with your hardware noted; a 1-sentence
pick recommendation. Expect HNSW to build slower but query faster with higher recall.

200 docs is far too small to see the production gap; the *methodology* is the deliverable -
swap in a 1M-row matrix when you have one. `algorithm='auto'` stands in for HNSW, `ball_tree`
for the IVF-shaped tree.

In [ ]:
import time

def time_build(algorithm: str, mat: np.ndarray, n_query: int = 50, metric: str = 'cosine') -> dict:
    t0 = time.perf_counter()
    nn = NearestNeighbors(n_neighbors=10, metric=metric, algorithm=algorithm).fit(mat)
    build_ms = (time.perf_counter() - t0) * 1000.0
    qs = mat[:n_query]
    t1 = time.perf_counter()
    nn.kneighbors(qs, n_neighbors=10)
    query_ms = (time.perf_counter() - t1) * 1000.0 / n_query
    return {'build_ms': build_ms, 'query_ms_avg': query_ms}

# 'auto' picks an HNSW-class structure for cosine on small matrices; 'ball_tree' acts as IVF-shaped surrogate
hnsw_like = time_build('auto', corpus_vecs, metric='cosine')
# ball_tree does not support the cosine metric in scikit-learn; on the L2-normalized
# embeddings used here euclidean nearest-neighbor ranking matches cosine, so we use
# euclidean for the IVF-like tree surrogate.
ivf_like = time_build('ball_tree', corpus_vecs, metric='euclidean')

print(f'HNSW-like (algorithm=auto)        build={hnsw_like["build_ms"]:.2f} ms  query={hnsw_like["query_ms_avg"]:.3f} ms/q')
print(f'IVF-like  (algorithm=ball_tree)   build={ivf_like["build_ms"]:.2f} ms  query={ivf_like["query_ms_avg"]:.3f} ms/q')
print('\nProduction read: on 12M chunks the HNSW build is hours, IVF is minutes. AcmeCorp with weekly reindex picks HNSW; daily reindex picks IVF or product-quantised IVF.')

---
## Exercise 3 (Medium, 5 pts) - Qdrant in Docker: insert and benchmark vs pgvector

**Task.** Run `docker run -p 6333:6333 qdrant/qdrant`. Use `qdrant-client` to create a
collection with 1536-dim cosine distance and HNSW (m=16, ef_construct=128). Insert the same
vectors from Ex2, run the same benchmark, and compare to pgvector HNSW.

**Success criteria** - collection up; vectors inserted; p50/p95/recall reported without
cherry-picking. Expect Qdrant p95 a few ms faster than pgvector HNSW on the same hardware.

`QdrantClient(":memory:")` runs a real Qdrant inside the process - same API as the
CoreWeave-hosted cluster, no Docker needed. If `qdrant-client` is missing, a small NumPy shim
keeps the cell runnable offline.

In [ ]:
# Strategy 2 - Qdrant in-memory client.
# Uses the REAL qdrant-client when it is installed; otherwise falls back to a small
# numpy-backed shim so the retrieval demo still runs fully offline.
import numpy as np

try:
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct
    _QDRANT_REAL = True
except Exception as _e:
    _QDRANT_REAL = False
    print(f'qdrant-client unavailable ({_e}); using offline numpy Qdrant shim')

if not _QDRANT_REAL:
    class Distance:
        COSINE = 'Cosine'
    class VectorParams:
        def __init__(self, size, distance):
            self.size, self.distance = size, distance
    class PointStruct:
        def __init__(self, id, vector, payload=None):
            self.id, self.vector, self.payload = id, vector, payload or {}
    class _QHit:
        def __init__(self, id, score, payload):
            self.id, self.score, self.payload = id, score, payload
    class _QCount:
        def __init__(self, count):
            self.count = count
    class QdrantClient:
        """Minimal offline stand-in for the subset of the API this notebook uses."""
        def __init__(self, *a, **k):
            self._store = {}
        def collection_exists(self, collection_name):
            return collection_name in self._store
        def delete_collection(self, collection_name):
            self._store.pop(collection_name, None)
        def create_collection(self, collection_name, vectors_config=None):
            self._store[collection_name] = {}
        def upsert(self, collection_name, points):
            s = self._store.setdefault(collection_name, {})
            for p in points:
                s[p.id] = (np.asarray(p.vector, dtype=np.float32), p.payload)
        def count(self, collection_name):
            return _QCount(len(self._store.get(collection_name, {})))
        def search(self, collection_name, query_vector, limit=5):
            s = self._store.get(collection_name, {})
            q = np.asarray(query_vector, dtype=np.float32)
            qn = float(np.linalg.norm(q)) + 1e-9
            hits = [_QHit(pid, float(v @ q / ((np.linalg.norm(v) + 1e-9) * qn)), payload)
                    for pid, (v, payload) in s.items()]
            hits.sort(key=lambda h: h.score, reverse=True)
            return hits[:limit]

qc = QdrantClient(':memory:')

_COLL = 'acmecorp_finserv'
_vp = VectorParams(size=int(corpus_vecs.shape[1]), distance=Distance.COSINE)
# recreate_collection is deprecated/removed in newer qdrant-client; prefer the
# collection_exists + create_collection path and fall back to recreate_collection.
if hasattr(qc, 'collection_exists'):
    if qc.collection_exists(_COLL):
        qc.delete_collection(_COLL)
    qc.create_collection(collection_name=_COLL, vectors_config=_vp)
else:
    qc.recreate_collection(collection_name=_COLL, vectors_config=_vp)

points = [
    PointStruct(id=i, vector=corpus_vecs[i].tolist(), payload={'text': corpus[i]})
    for i in range(len(corpus))
]
qc.upsert(collection_name=_COLL, points=points)
print(f'Qdrant collection has {qc.count(collection_name=_COLL).count} points')

def qdrant_search(query_vector, limit=5):
    """Version-tolerant Qdrant search: uses .search when present (older clients / shim),
    else the newer .query_points API. Returns a list of hit objects with .id/.score/.payload."""
    if hasattr(qc, 'search'):
        return qc.search(collection_name=_COLL, query_vector=query_vector, limit=limit)
    return qc.query_points(collection_name=_COLL, query=query_vector, limit=limit).points

hits = qdrant_search(qvec.tolist(), limit=5)
print(f'\nQdrant top-5 for: "{query}"')
for h in hits:
    print(f'  id={h.id:3d} score={h.score:.3f} :: {h.payload["text"][:90]}...')


---
## Exercise 4 (Medium, 5 pts) - pgvector hybrid: vector + tsvector + RRF

**Task.** Add a generated `tsv tsvector` column, build a GIN index, and write a single SQL
query that runs vector ANN top-20 + BM25 top-20 (`ts_rank_cd`) and fuses them with RRF (k=60)
in a CTE. Return top-10.

**Success criteria** - one fused top-10; an exact-match query surfaces the literal section doc
that pure vector missed. Dense alone misses the literal string; keyword alone misses the
paraphrases; RRF gets both.

The production SQL (the `WITH vec AS (...), bm AS (...), rrf AS (...)` CTE) is on the HTML page.
Below is the same fusion in-process - cosine top-20, a term-frequency keyword top-20, then RRF.

In [ ]:
def cosine_top_n(qvec: np.ndarray, n: int = 20):
    sims = corpus_vecs @ qvec / (np.linalg.norm(corpus_vecs, axis=1) * np.linalg.norm(qvec) + 1e-9)
    return list(np.argsort(-sims)[:n])

def keyword_top_n(query: str, n: int = 20):
    terms = [t.lower() for t in query.split() if len(t) > 2]
    scores = []
    for i, doc in enumerate(corpus):
        tokens = doc.lower().split()
        scores.append(sum(tokens.count(t) for t in terms))
    return list(np.argsort(-np.array(scores))[:n])

def rrf(rankings, k: int = 60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[int(doc_id)] = scores.get(int(doc_id), 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

hybrid_query = 'GDPR Article 44 cross-border transfer'
hqvec = embed(hybrid_query)
dense_ranking = cosine_top_n(hqvec, n=20)
kw_ranking = keyword_top_n(hybrid_query, n=20)
fused = rrf([dense_ranking, kw_ranking])[:5]

print(f'Hybrid pgvector-sim top-5 for: "{hybrid_query}"')
for i in fused:
    print(f'  id={i:3d} :: {corpus[i][:90]}...')
print('\nDense alone can miss the literal "Article 44" string; keyword alone misses the cross-border-residency paraphrases. RRF gets both.')

---
## Exercise 5 (Hard, 5 pts) - Quantization: recall vs memory footprint

**Task.** Full-precision fp32 vectors are the biggest line item in a 5M-chunk index. Cut the
footprint two ways and measure the recall you pay for it: pgvector `halfvec` (fp16, ~2x) and
int8 **scalar quantization** (~4x, with rescoring). Measure `recall@10` against the fp32 HNSW
index and report each memory footprint.

**Success criteria** - a quantized variant built; recall@10 measured against the fp32 baseline;
memory footprint reported for full-precision vs quantized. Plot or state where you draw the line
for a 5M-chunk index.

The worked solution does int8 scalar **and** binary quantization in pure NumPy (offline), so you
see the full lever: int8 is ~free (4x smaller, recall barely moves); binary is 32x smaller but
leans on a float rescoring pass to claw recall back.

In [ ]:
# Vector quantization in pure NumPy: int8 scalar + binary (with float rescoring).
# Ground truth = exact full-precision cosine top-10. Everything runs offline, no key.
import numpy as np

X = corpus_vecs.astype(np.float32)          # (N, 1536), already L2-normalized
N, DIM = X.shape

def _normalize_rows(m):
    return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)

# Use a deterministic handful of in-corpus vectors as queries (offline).
rng = np.random.default_rng(0)
query_ids = rng.choice(N, size=25, replace=False)
Q = X[query_ids]

def top_k(qmat, mat, k=10):
    """Cosine top-k of each row of qmat against mat, excluding the self-match."""
    sims = _normalize_rows(qmat) @ _normalize_rows(mat).T
    for row, qid in enumerate(query_ids):
        sims[row, qid] = -np.inf
    return np.argsort(-sims, axis=1)[:, :k]

gt = top_k(Q, X, k=10)   # (25, 10) full-precision ground-truth ids

# --- (a) int8 SCALAR quantization: per-dim min/max -> 1 byte, dequantize for scoring ---
lo = X.min(axis=0)
hi = X.max(axis=0)
scale = (hi - lo) / 255.0
scale[scale == 0.0] = 1.0
codes = np.round((X - lo) / scale).clip(0, 255).astype(np.uint8)   # 1 byte / dim
X_int8 = lo + codes.astype(np.float32) * scale                     # dequantized
q_codes = np.round((Q - lo) / scale).clip(0, 255).astype(np.uint8)
Q_int8 = lo + q_codes.astype(np.float32) * scale
int8_top = top_k(Q_int8, X_int8, k=10)

# --- (b) BINARY quantization: sign bits + Hamming distance, then float RESCORING ---
packed = np.packbits(X > 0.0, axis=1)       # (N, DIM/8) uint8 -> 1 bit / dim
q_packed = np.packbits(Q > 0.0, axis=1)
POPCOUNT = np.array([bin(b).count('1') for b in range(256)], dtype=np.int32)

def hamming_top_n(qp, n):
    dist = POPCOUNT[np.bitwise_xor(qp[None, :], packed)].sum(axis=1)
    return np.argsort(dist)[:n]

RESCORE_N = 50
binary_top = np.empty((len(query_ids), 10), dtype=int)
for row in range(len(query_ids)):
    cand = hamming_top_n(q_packed[row], RESCORE_N)   # wide shortlist by Hamming
    cand = cand[cand != query_ids[row]]              # drop the self-match
    cand_sims = _normalize_rows(Q[row:row + 1]) @ _normalize_rows(X[cand]).T  # float rescoring
    binary_top[row] = cand[np.argsort(-cand_sims[0])[:10]]

# --- recall@10 vs full precision ---
def recall_at_k(approx, truth):
    hits = [len(set(a.tolist()) & set(t.tolist())) for a, t in zip(approx, truth)]
    return float(np.mean(hits)) / truth.shape[1]

r_int8 = recall_at_k(int8_top, gt)
r_bin = recall_at_k(binary_top, gt)

# --- memory footprint (bytes / vector) ---
f32_bytes = 4 * DIM
int8_bytes = 1 * DIM
bin_bytes = DIM // 8

print(f'corpus: {N} vectors x {DIM} dims, {len(query_ids)} probe queries\n')
print(f'{"scheme":<20}{"bytes/vec":>11}{"vs float32":>14}{"recall@10":>12}')
print(f'{"float32 (full)":<20}{f32_bytes:>11}{"1x":>14}{1.000:>12.3f}')
print(f'{"int8 scalar":<20}{int8_bytes:>11}{f"{f32_bytes // int8_bytes}x smaller":>14}{r_int8:>12.3f}')
print(f'{"binary + rescore":<20}{bin_bytes:>11}{f"{f32_bytes // bin_bytes}x smaller":>14}{r_bin:>12.3f}')
print('\nRead: int8 is the near-free lunch (4x smaller, recall ~unchanged). Binary is 32x smaller')
print('but leans on the float rescoring pass to claw recall back - widen RESCORE_N to trade latency for recall.')

---
## Exercise 6 (Medium, 5 pts) - Filtered search: tenant_id isolation, pre-filter vs post-filter

**Task.** AcmeCorp runs one shared index across 50 business units, and every query must return
only the caller's tenant. Tag each row with a `tenant_id`, then run a filtered ANN search two
ways - **post-filter** (ANN first, drop wrong tenants afterward) and **pre-filter** (the filter
is applied inside the graph traversal). Compare how many in-tenant rows each returns and verify
**zero cross-tenant leakage**.

**Success criteria** - both filter modes run; a selective tenant starves the post-filter page
while pre-filter always fills `k`; every returned row carries the requested `tenant_id` (one
cross-tenant row is a data-isolation incident, not a recall miss).

Production SQL: pgvector 0.8 iterative index scans (`WHERE tenant_id = $1 ORDER BY embedding
<=> $2`) and Qdrant filterable HNSW keep recall up. The worked solution below is the same
pre/post contract in NumPy.

In [ ]:
import numpy as np

# AcmeCorp runs ONE shared index across 50 business units; every query must return only the
# caller's tenant. Tag each row (id % 50), exactly like the exercise's UPDATE ... SET tenant_id.
tenant_ids = np.array([i % 50 for i in range(len(corpus))])
target_tenant = 7

q_text = 'GDPR cross-border data transfer and fraud reporting thresholds'
qv = embed(q_text)
norm = corpus_vecs / (np.linalg.norm(corpus_vecs, axis=1, keepdims=True) + 1e-9)
sims = norm @ (qv / (np.linalg.norm(qv) + 1e-9))
K = 10

# POST-filter: ANN top-K over the whole index first, then drop rows from other tenants.
ann_order = np.argsort(-sims)
post = [int(i) for i in ann_order[:K] if tenant_ids[i] == target_tenant]

# PRE-filter: restrict to the tenant's rows first (filter inside the traversal), then rank.
allowed = np.where(tenant_ids == target_tenant)[0]
pre = [int(i) for i in allowed[np.argsort(-sims[allowed])][:K]]

print(f'tenant {target_tenant}: {len(allowed)} of {len(corpus)} rows are in-tenant\n')
print(f'POST-filter returned {len(post)}/{K} in-tenant rows (a selective tenant starves the page)')
print(f'PRE-filter  returned {len(pre)}/{K} in-tenant rows (always fills k)\n')

# Zero cross-tenant leakage is a hard gate: one wrong-tenant row is a data-isolation incident.
assert all(tenant_ids[i] == target_tenant for i in pre), 'cross-tenant leakage in pre-filter!'
assert all(tenant_ids[i] == target_tenant for i in post), 'cross-tenant leakage in post-filter!'
print('OK - zero cross-tenant leakage. Pre-filter holds recall; post-filter under-fills k on a')
print('selective predicate. On a real store you flip one flag - the trap is a POC that silently')
print('post-filters and quietly returns half-empty pages for small tenants.')


---
## Exercise 7 (Hard, 5 pts) - pgvector to Qdrant migration runbook

**Task.** Draft the migration runbook an AcmeCorp SRE would execute. Submit a markdown doc with
5 phases; each phase has goal, commands, success criteria, rollback:

1. **Dual-write build** - app-layer writes embeddings to both pgvector and Qdrant; verify count
   parity hourly.
2. **Backfill** - bulk-copy 10M existing vectors into Qdrant; verify recall@10 on a 200-question
   golden set within 1pp of pgvector.
3. **Shadow reads** - mirror every read to Qdrant for 48h; log top-10 diff; tune HNSW params.
4. **Canary cutover** - 5% -> 25% -> 100% read traffic over 5 days, feature-flag controlled.
5. **Cleanup** - stop dual-write after the 14-day rollback window; drop the pgvector index;
   release disk.

**Success criteria** - all 5 phases present; a rollback for each ("if X then revert by Y in
under 60 seconds"); a golden-set recall check; a 60-second flag-flip rollback in the cutover
phase.

The load-bearing primitive is **dual-write with a read-consistency check** - the worked solution
shows exactly that contract (write to both stores, then confirm the new id surfaces in both).

In [ ]:
def dual_write(doc_id: int, text: str, vec: np.ndarray) -> dict:
    """Write to both pgvector-sim and qdrant-sim. Return per-target outcomes."""
    out = {}
    try:
        pg.upsert(doc_id, text, vec)
        out['pgvector_sim'] = 'ok'
    except Exception as e:
        out['pgvector_sim'] = f'error: {e}'
    try:
        qc.upsert(
            collection_name=_COLL,
            points=[PointStruct(id=doc_id, vector=vec.tolist(), payload={'text': text})],
        )
        out['qdrant'] = 'ok'
    except Exception as e:
        out['qdrant'] = f'error: {e}'
    return out

new_text = 'GDPR Article 35 - the data controller must run a DPIA covering any high-risk AI-assisted credit-underwriting model before deployment.'
new_vec = embed(new_text)
result = dual_write(doc_id=9999, text=new_text, vec=new_vec)
print('Dual-write result:', result)

# Read-consistency check
pg_hits = [r[0] for r in pg.query_cosine(new_vec, top_k=3)]
qd_hits = [h.id for h in qdrant_search(new_vec.tolist(), limit=3)]
print(f'pgvector-sim top-3 ids: {pg_hits}')
print(f'qdrant       top-3 ids: {qd_hits}')
print(f'Consistency (id 9999 in both): {9999 in pg_hits and 9999 in qd_hits}')

---
## Exercise 8 (GRADED, 5 pts + 1 bonus) - Pick a vector store for AcmeCorp RAG

**Task.** Submit a 1-page decision memo. The customer: AcmeCorp. The corpus: 800 OCC bulletins
+ EU regulatory notices, growing to ~5M chunks. Constraints: GDPR residency (CoreWeave EU
region), 500ms end-to-end SLO, existing Postgres on CoreWeave. Deliverables:

1. **(2 pts) 5-store decision matrix** - pgvector, Qdrant, Pinecone, Weaviate, Milvus. Rows =
   constraints (residency, latency p95, native hybrid, ops burden, $/month at 5M vectors, vendor
   onboarding weeks). Fill every cell with a number or a 1-word verdict.
2. **(1 pt) Latency + cost summary** - p50/p95 for each store at 5M vectors; $/month at 100K QPS.
3. **(1 pt) GDPR analysis** - residency, cross-border DPA risk, audit trail, IAM. 4 bullets.
4. **(1 pt) Recommendation memo** - 1 paragraph to the AcmeCorp CTO + CISO; explicitly address
   the "but Pinecone benchmarked 6ms faster" question.
5. **(+1 bonus) Rollback plan** - the 5-line runbook to revert to the second choice in 4 hours.

**Reference decision matrix** (the shape you paste into the architecture deck):

| Store | Strength | Weakness | Pick when... |
|-------|----------|----------|--------------|
| pgvector | Already in the customer Postgres; transactional joins; no new vendor | Slower past ~50M vectors; manual HNSW tuning | AcmeCorp / any financial-services shop with Postgres in prod and a procurement clock ticking |
| Qdrant (self-hosted, CoreWeave) | Best price/perf at scale; runs in-region; native quantization + payload filtering | You operate it (on-call, snapshots, upgrades) | Data-sovereign teams already running Kubernetes |
| Qdrant Cloud | Managed EU/US; native quantization + filterable HNSW | Cross-border flow needs a GDPR Chapter V review | Non-PII corpora, dev sandboxes |
| Pinecone | Zero ops; serverless default; fastest dev velocity | ~$50/mo floor; egress ~$0.08-0.10/GB; hard to migrate off | Greenfield, non-regulated corpora, fast POCs |
| Weaviate / Milvus | OSS; hybrid built in; Milvus adds GPU ANN (CAGRA on NVIDIA cuVS) | More operator surface | Teams needing hybrid or GPU ANN with ops headroom |

The gradeable primitive is a `pick_store(constraints)` function that encodes the FDE's rule of
thumb - GDPR-residency-first customers lead with pgvector or Qdrant-on-CoreWeave; time-to-demo
customers lead with Pinecone or Qdrant Cloud.

In [ ]:
def pick_store(constraints: dict) -> dict:
    """Encode the FDE store-picker rule of thumb as a pure function."""
    if constraints.get('residency') == 'eu' and constraints.get('postgres_in_prod'):
        return {'store': 'pgvector', 'why': 'reuse existing Postgres, no new vendor, GDPR-clean'}
    if constraints.get('residency') == 'eu' and constraints.get('scale_vectors', 0) > 50_000_000:
        return {'store': 'qdrant_self_hosted_coreweave',
                'why': '>50M vectors needs purpose-built ANN; CoreWeave keeps it in-region'}
    if constraints.get('time_to_demo_days', 999) <= 14 and not constraints.get('pii'):
        return {'store': 'pinecone',
                'why': 'serverless is fastest to a demo; no PII so cross-border is fine for a POC'}
    return {'store': 'qdrant_cloud', 'why': 'managed, hybrid-ready, accept EU residency for now'}

examples = [
    {'residency': 'eu',  'postgres_in_prod': True,  'scale_vectors': 12_000_000, 'pii': True,  'time_to_demo_days': 42},
    {'residency': 'eu',  'postgres_in_prod': False, 'scale_vectors': 80_000_000, 'pii': True,  'time_to_demo_days': 60},
    {'residency': 'any', 'postgres_in_prod': False, 'scale_vectors': 500_000,    'pii': False, 'time_to_demo_days': 10},
    {'residency': 'eu',  'postgres_in_prod': False, 'scale_vectors': 2_000_000,  'pii': False, 'time_to_demo_days': 21},
]
for c in examples:
    print(c, '->', pick_store(c))

# Stretch (grade the memo, not just the router):
# 1. Add a compliance_review_weeks constraint; route long-clock customers to what they run today.
# 2. Encode the GDPR Chapter V cross-border-transfer test explicitly.
# 3. Return a cost band (low/medium/high) justified by the per-query pricing you measured above.


---
## Grading summary - 40 pts total

Pass = 28/40. Distinction = 36/40. Reviewers watch for these 4 common pitfalls:

- **Recommending Pinecone for financial services without GDPR analysis** - the 6ms latency win
  does not survive the procurement stack. Auto-deduct 2 pts.
- **Treating index rebuild as a 1-click op** - HNSW rebuild on 10M vectors is a 4-8h ops event
  with a runbook, not a CLI command. Auto-deduct 2 pts.
- **No rollback in the migration plan** - dual-write without a 60-second feature-flag revert is
  malpractice. Auto-deduct 3 pts.
- **Dishonest measurement** - if the Ex2/Ex3 benchmark numbers do not match the printed output,
  auto-deduct 5 pts.

**Submission checklist**
- [ ] Ex1-Ex6 - each cell runs top to bottom and prints the expected shape
- [ ] Ex5 - recall@10 measured against the fp32 baseline, bytes/vector reported
- [ ] Ex6 - zero cross-tenant leakage asserted for both filter modes
- [ ] Ex7 - 5-phase runbook with a rollback for each phase and a 60-second cutover flag-flip
- [ ] Ex8 - 5-store matrix, latency+cost summary, GDPR analysis, recommendation memo (+ rollback bonus)

**Feedback SLA** - submit your solution in the cohort Discord `#practice-labs` channel.
Rubric-graded feedback is returned within **24 hours** (Plus-tier mentor review); self-paced
learners use this worked notebook and the rubric above to self-assess right away.